# 603 — Cross-Screen Replication

## Objective

Evaluate whether the frozen notebook-600 GDSC program–drug associations that satisfied the prospectively defined FDR criterion show directionally concordant cross-screen pharmacogenomic evidence for the same exact compounds in CTRP and/or PRISM where frozen compound, screen, lineage, and response coverage permit evaluation.

This notebook treats cross-screen agreement as external pharmacogenomic replication of resistance-like computational associations. It does not establish independent biological validation, clinical drug resistance, therapeutic efficacy, causal drug-response mechanisms, validated biomarkers, or validated therapeutic targets.

## Analytical boundary

Notebook 603 is restricted to the frozen notebook-600 GDSC `drug × consensus program` associations with `q_GDSC < 0.05`.

The notebook consumes the registered notebook-600 handoffs as authoritative inputs. It does not:

* refit the GDSC developmental/internal association models;
* reconstruct frozen cell-line or exact-compound mappings;
* reproject or restandardize the frozen consensus-program scores;
* re-aggregate CTRP repeated experiments;
* reselect PRISM primary screens;
* recompute frozen full-screen drug eligibility;
* use fuzzy, target-based, mechanism-based, drug-family, or manual synonym matching to rescue compound identity; or
* use notebook-601 predictive performance or notebook-602 attribution to select or redefine replication hypotheses.

Primary exact-compound replication uses the frozen resource-specific response representations and the same lineage-adjusted association structure used in notebook 600:

`response ~ program_score + C(OncotreeLineage)`

with HC3 heteroskedasticity-robust inference and two-sided tests.

CTRP and PRISM define separate external multiplicity families. A screen-level association is classified as replicated only when the external result satisfies both `q_external < 0.05` and direction concordance with the frozen GDSC coefficient.

The prospectively frozen notebook-603 specification was established on `2026-09-22`, before inspection of any CTRP or PRISM replication coefficient, p-value, q-value, direction-concordance result, overlap-specific result, or replication status.

## Overlap-aware replication framework

The primary external analysis uses the complete frozen eligible analytical set for the corresponding exact compound in each external screen.

Because GDSC, CTRP, and PRISM may contain overlapping cell-line models, cell-line overlap remains explicit throughout notebook 603. Cross-screen agreement involving shared models is not automatically interpreted as independent cell-level replication.

A prespecified secondary analysis evaluates external models not represented in the corresponding GDSC primary analytical set. The same frozen `20 / 3 / 100` lineage-support rule is applied to this non-overlapping-model subset.

Insufficient non-overlap coverage is retained as `INSUFFICIENT_NONOVERLAP_COVERAGE` rather than interpreted as failed replication. Favorable non-overlap results may strengthen an already replicated primary result but cannot rescue a failed primary full-screen replication.

Shared-model and lineage-composition analyses remain descriptive diagnostics and do not create alternative replication routes.

## Evidence isolation

* **GDSC:** developmental/internal reference supplying the frozen notebook-600 association hypothesis and direction.
* **CTRP:** external cross-screen pharmacogenomic replication resource.
* **PRISM:** external cross-screen pharmacogenomic replication resource under the frozen primary-screen hierarchy.
* **Notebook 601:** predictive-validity evidence remains separate from primary replication-hypothesis selection, multiplicity, and replication status.
* **Notebook 602:** program-level attribution and stability evidence remains separate from primary replication decisions and may be appended only after replication results have been determined.
* **Drug family, target, and mechanism annotations:** contextual pharmacological evidence only; they do not substitute for exact-compound replication.

Hypotheses without exact-compound correspondence, a valid primary screen, or sufficient frozen analytical coverage remain explicitly not evaluable rather than being counted as biological replication failures.

Native response scales remain resource-specific. Cross-screen effect-size comparison is descriptive and does not justify pooling GDSC, CTRP, and PRISM responses or treating their coefficients as numerically interchangeable.

Residual proliferation, platform-specific assay differences, shared upstream molecular representations, cell-line overlap, and other unresolved biological or technical confounding remain explicit limitations.

Cross-screen replication, non-overlapping-model corroboration, predictive validity, model attribution, biological contextualization, functional-vulnerability evidence, and later perturbational evidence remain distinct evidence dimensions.


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [ ]:
# =============================================================================
# Resolve frozen notebook-600 handoffs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase6.600.program_score_universe",
    "phase6.600.ctrp_model_crosswalk",
    "phase6.600.cross_resource_compound_catalog",
    "phase6.600.drug_eligibility",
    "phase6.600.gdsc_analysis_universe",
    "phase6.600.ctrp_analysis_universe",
    "phase6.600.prism_analysis_universe",
    "phase6.600.gdsc_program_drug_associations",
    "phase6.600.analysis_metadata",
)

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

for artifact_id, path in input_paths.items():
    print(f"{artifact_id}: {project_relative_path(path)}")

In [ ]:
# =============================================================================
# Load frozen notebook-600 replication inputs
# =============================================================================

program_scores = pd.read_parquet(
    input_paths["phase6.600.program_score_universe"]
)

ctrp_model_crosswalk = pd.read_csv(
    input_paths["phase6.600.ctrp_model_crosswalk"]
)

compound_catalog = pd.read_csv(
    input_paths["phase6.600.cross_resource_compound_catalog"]
)

drug_eligibility = pd.read_csv(
    input_paths["phase6.600.drug_eligibility"]
)

gdsc_universe = pd.read_parquet(
    input_paths["phase6.600.gdsc_analysis_universe"]
)

ctrp_universe = pd.read_parquet(
    input_paths["phase6.600.ctrp_analysis_universe"]
)

prism_universe = pd.read_parquet(
    input_paths["phase6.600.prism_analysis_universe"]
)

gdsc_associations = pd.read_parquet(
    input_paths["phase6.600.gdsc_program_drug_associations"]
)

with open(
    input_paths["phase6.600.analysis_metadata"],
    encoding="utf-8",
) as handle:
    notebook600_metadata = json.load(handle)

print("Program-score universe shape:", program_scores.shape)
print("CTRP model crosswalk shape:", ctrp_model_crosswalk.shape)
print("Exact-compound catalog shape:", compound_catalog.shape)
print("Drug eligibility shape:", drug_eligibility.shape)
print("GDSC analysis universe shape:", gdsc_universe.shape)
print("CTRP analysis universe shape:", ctrp_universe.shape)
print("PRISM analysis universe shape:", prism_universe.shape)
print("GDSC association results shape:", gdsc_associations.shape)

In [ ]:
# =============================================================================
# Define the frozen GDSC replication hypothesis set
# =============================================================================

gdsc_replication_hypotheses = (
    gdsc_associations.loc[
        gdsc_associations["fdr_05"].astype(bool),
        [
            "DRUG_ID",
            "DRUG_NAME",
            "program",
            "beta",
            "p_value",
            "q_value",
            "association_direction",
        ],
    ]
    .rename(
        columns={
            "DRUG_ID": "gdsc_drug_id",
            "beta": "gdsc_beta",
            "p_value": "gdsc_p_value",
            "q_value": "gdsc_q_value",
            "association_direction": "gdsc_direction",
        }
    )
    .sort_values(
        [
            "gdsc_drug_id",
            "program",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Frozen GDSC replication hypotheses:",
    len(gdsc_replication_hypotheses),
)
print(
    "Unique GDSC drugs:",
    gdsc_replication_hypotheses["gdsc_drug_id"].nunique(),
)
print(
    "Programs represented:",
    gdsc_replication_hypotheses["program"].nunique(),
)

display(gdsc_replication_hypotheses.head())

In [ ]:
# =============================================================================
# Attach frozen exact-compound identities
# =============================================================================

exact_compound_identity = (
    compound_catalog.loc[
        compound_catalog["gdsc_drug_id"].notna(),
        [
            "drug_name_key",
            "gdsc_drug_id",
            "ctrp_master_cpd_id",
            "ctrp_drug_name",
            "prism_broad_id",
            "prism_drug_name",
            "resources_present",
            "screen_pattern",
            "primary_prism_screen",
        ],
    ]
    .copy()
)

exact_compound_identity["gdsc_drug_id"] = (
    exact_compound_identity["gdsc_drug_id"].astype(int)
)

replication_hypotheses = (
    gdsc_replication_hypotheses
    .merge(
        exact_compound_identity,
        on="gdsc_drug_id",
        how="left",
        validate="many_to_one",
    )
)

has_ctrp_match = replication_hypotheses[
    "ctrp_master_cpd_id"
].notna()

has_prism_match = replication_hypotheses[
    "prism_broad_id"
].notna()

print(
    "Replication hypotheses:",
    len(replication_hypotheses),
)
print(
    "With exact CTRP compound match:",
    has_ctrp_match.sum(),
)
print(
    "With exact PRISM compound match:",
    has_prism_match.sum(),
)
print(
    "With exact match in both external screens:",
    (has_ctrp_match & has_prism_match).sum(),
)
print(
    "Without exact external compound match:",
    (~has_ctrp_match & ~has_prism_match).sum(),
)

display(replication_hypotheses.head())

In [ ]:
# =============================================================================
# Expand hypotheses by external resource
# =============================================================================

manifest_columns = [
    "gdsc_drug_id",
    "DRUG_NAME",
    "program",
    "gdsc_beta",
    "gdsc_p_value",
    "gdsc_q_value",
    "gdsc_direction",
    "drug_name_key",
]

ctrp_hypotheses = (
    replication_hypotheses
    .assign(
        external_resource="CTRP",
        external_drug_id=lambda data: (
            data["ctrp_master_cpd_id"]
            .astype("Int64")
            .astype("string")
        ),
        primary_screen=pd.NA,
    )
    [
        manifest_columns
        + [
            "external_resource",
            "external_drug_id",
            "primary_screen",
        ]
    ]
)

prism_hypotheses = (
    replication_hypotheses
    .assign(
        external_resource="PRISM",
        external_drug_id=lambda data: (
            data["prism_broad_id"].astype("string")
        ),
        primary_screen=lambda data: (
            data["primary_prism_screen"]
        ),
    )
    [
        manifest_columns
        + [
            "external_resource",
            "external_drug_id",
            "primary_screen",
        ]
    ]
)

replication_manifest = pd.concat(
    [
        ctrp_hypotheses,
        prism_hypotheses,
    ],
    ignore_index=True,
)

print(
    "Resource-level hypothesis rows:",
    len(replication_manifest),
)

In [ ]:
# =============================================================================
# Attach frozen external eligibility
# =============================================================================

external_eligibility = (
    drug_eligibility.loc[
        drug_eligibility["resource"].isin(
            [
                "CTRP",
                "PRISM_exact_cross_resource",
            ]
        )
    ]
    .assign(
        external_resource=lambda data: (
            data["resource"].replace(
                {
                    "PRISM_exact_cross_resource": "PRISM",
                }
            )
        ),
        external_drug_id=lambda data: (
            data["resource_drug_id"].astype("string")
        ),
    )
    .rename(
        columns={
            "supported_lineages": "frozen_supported_lineages",
            "supported_models": "frozen_supported_models",
            "eligible": "frozen_fullscreen_eligible",
            "analysis_universe": "frozen_analysis_universe",
        }
    )
    [
        [
            "external_resource",
            "external_drug_id",
            "frozen_supported_lineages",
            "frozen_supported_models",
            "frozen_fullscreen_eligible",
            "frozen_analysis_universe",
        ]
    ]
)

replication_manifest = (
    replication_manifest
    .merge(
        external_eligibility,
        on=[
            "external_resource",
            "external_drug_id",
        ],
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "gdsc_drug_id",
            "program",
            "external_resource",
        ]
    )
    .reset_index(drop=True)
)

display(
    replication_manifest.groupby(
        "external_resource",
        dropna=False,
    )
    .agg(
        hypotheses=("program", "size"),
        exact_matches=("external_drug_id", "count"),
        frozen_eligible=("frozen_fullscreen_eligible", "sum"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Assign frozen primary evaluability status
# =============================================================================

VALID_PRISM_PRIMARY_SCREENS = {
    "HTS002",
    "MTS006",
    "MTS010",
}

has_exact_match = replication_manifest[
    "external_drug_id"
].notna()

has_valid_primary_screen = (
    replication_manifest["external_resource"].eq("CTRP")
    | replication_manifest["primary_screen"].isin(
        VALID_PRISM_PRIMARY_SCREENS
    )
)

has_frozen_coverage = replication_manifest[
    "frozen_fullscreen_eligible"
].eq(True)

replication_manifest["preanalysis_status"] = np.select(
    [
        ~has_exact_match,
        has_exact_match & ~has_valid_primary_screen,
        has_exact_match
        & has_valid_primary_screen
        & ~has_frozen_coverage,
    ],
    [
        "NOT_EVALUABLE_NO_EXACT_MATCH",
        "NOT_EVALUABLE_PRIMARY_SCREEN",
        "NOT_EVALUABLE_COVERAGE",
    ],
    default="EVALUABLE_PRIMARY",
)

replication_manifest["primary_evaluable"] = (
    replication_manifest["preanalysis_status"]
    .eq("EVALUABLE_PRIMARY")
)

display(
    replication_manifest.groupby(
        [
            "external_resource",
            "preanalysis_status",
        ],
        dropna=False,
    )
    .size()
    .rename("hypotheses")
    .reset_index()
)

print(
    "Primary evaluable hypotheses:",
    replication_manifest["primary_evaluable"].sum(),
)

In [ ]:
# =============================================================================
# Build resource-specific primary model sets
# =============================================================================

gdsc_model_sets = (
    gdsc_universe.groupby("DRUG_ID")["ModelID"]
    .agg(lambda values: frozenset(values))
    .to_dict()
)

ctrp_model_sets = (
    ctrp_universe.groupby("master_cpd_id")["ModelID"]
    .agg(lambda values: frozenset(values))
    .to_dict()
)

prism_model_sets = (
    prism_universe.groupby("broad_id")["ModelID"]
    .agg(lambda values: frozenset(values))
    .to_dict()
)


def get_external_model_set(row):
    if not row["primary_evaluable"]:
        return frozenset()

    if row["external_resource"] == "CTRP":
        return ctrp_model_sets.get(
            int(row["external_drug_id"]),
            frozenset(),
        )

    return prism_model_sets.get(
        row["external_drug_id"],
        frozenset(),
    )

In [ ]:
# =============================================================================
# Quantify GDSC–external cell-line overlap
# =============================================================================

def summarize_gdsc_external_overlap(row):
    if not row["primary_evaluable"]:
        return pd.Series(
            {
                "gdsc_primary_models": pd.NA,
                "external_primary_models": pd.NA,
                "shared_models": pd.NA,
                "external_nonoverlap_models": pd.NA,
                "shared_fraction_external": np.nan,
            }
        )

    gdsc_models = gdsc_model_sets[row["gdsc_drug_id"]]
    external_models = get_external_model_set(row)

    shared_models = gdsc_models & external_models
    external_nonoverlap = external_models - gdsc_models

    return pd.Series(
        {
            "gdsc_primary_models": len(gdsc_models),
            "external_primary_models": len(external_models),
            "shared_models": len(shared_models),
            "external_nonoverlap_models": len(external_nonoverlap),
            "shared_fraction_external": (
                len(shared_models) / len(external_models)
            ),
        }
    )


overlap_summary = replication_manifest.apply(
    summarize_gdsc_external_overlap,
    axis=1,
)

replication_manifest = pd.concat(
    [
        replication_manifest,
        overlap_summary,
    ],
    axis=1,
)

display(
    replication_manifest.loc[
        replication_manifest["primary_evaluable"]
    ]
    .groupby("external_resource")
    .agg(
        hypotheses=("program", "size"),
        median_external_models=("external_primary_models", "median"),
        median_shared_models=("shared_models", "median"),
        median_nonoverlap_models=("external_nonoverlap_models", "median"),
        median_shared_fraction=("shared_fraction_external", "median"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Determine non-overlapping-model eligibility under the frozen 20 / 3 / 100 rule
# =============================================================================

evaluable_compounds = (
    replication_manifest.loc[
        replication_manifest["primary_evaluable"],
        [
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
        ],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


def summarize_nonoverlap_eligibility(row):
    gdsc_models = gdsc_model_sets[row["gdsc_drug_id"]]

    if row["external_resource"] == "CTRP":
        external_data = ctrp_universe.loc[
            ctrp_universe["master_cpd_id"].eq(
                int(row["external_drug_id"])
            )
        ].copy()
    else:
        external_data = prism_universe.loc[
            prism_universe["broad_id"].eq(
                row["external_drug_id"]
            )
        ].copy()

    nonoverlap_data = external_data.loc[
        ~external_data["ModelID"].isin(gdsc_models)
    ].copy()

    lineage_counts = (
        nonoverlap_data.groupby("OncotreeLineage")
        .size()
    )

    supported_lineages = lineage_counts.loc[
        lineage_counts >= 20
    ].index

    supported_nonoverlap = nonoverlap_data.loc[
        nonoverlap_data["OncotreeLineage"].isin(
            supported_lineages
        )
    ]

    n_supported_lineages = len(supported_lineages)
    n_supported_models = len(supported_nonoverlap)

    return pd.Series(
        {
            "nonoverlap_supported_lineages": n_supported_lineages,
            "nonoverlap_supported_models": n_supported_models,
            "nonoverlap_eligible": (
                n_supported_lineages >= 3
                and n_supported_models >= 100
            ),
        }
    )


nonoverlap_eligibility = pd.concat(
    [
        evaluable_compounds,
        evaluable_compounds.apply(
            summarize_nonoverlap_eligibility,
            axis=1,
        ),
    ],
    axis=1,
)

replication_manifest = (
    replication_manifest
    .merge(
        nonoverlap_eligibility,
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
        ],
        how="left",
        validate="many_to_one",
    )
)

display(
    replication_manifest.loc[
        replication_manifest["primary_evaluable"]
    ]
    .groupby("external_resource")
    .agg(
        primary_evaluable=("program", "size"),
        nonoverlap_evaluable=("nonoverlap_eligible", "sum"),
        median_nonoverlap_supported_models=(
            "nonoverlap_supported_models",
            "median",
        ),
        median_nonoverlap_supported_lineages=(
            "nonoverlap_supported_lineages",
            "median",
        ),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Assign non-overlapping-model evaluability status
# =============================================================================

replication_manifest["nonoverlap_status"] = pd.Series(
    pd.NA,
    index=replication_manifest.index,
    dtype="string",
)

primary_mask = replication_manifest["primary_evaluable"]

replication_manifest.loc[
    primary_mask
    & replication_manifest["nonoverlap_eligible"].eq(True),
    "nonoverlap_status",
] = "EVALUABLE_NONOVERLAP"

replication_manifest.loc[
    primary_mask
    & ~replication_manifest["nonoverlap_eligible"].eq(True),
    "nonoverlap_status",
] = "INSUFFICIENT_NONOVERLAP_COVERAGE"

display(
    replication_manifest.loc[
        replication_manifest["primary_evaluable"]
    ]
    .groupby(
        [
            "external_resource",
            "nonoverlap_status",
        ],
        dropna=False,
    )
    .size()
    .rename("hypotheses")
    .reset_index()
)

In [ ]:
# =============================================================================
# Build the dual-screen evaluable compound map
# =============================================================================

dual_screen_hypotheses = (
    replication_manifest.loc[
        replication_manifest["primary_evaluable"],
        [
            "gdsc_drug_id",
            "program",
            "external_resource",
            "external_drug_id",
        ],
    ]
    .pivot(
        index=[
            "gdsc_drug_id",
            "program",
        ],
        columns="external_resource",
        values="external_drug_id",
    )
    .dropna(
        subset=[
            "CTRP",
            "PRISM",
        ]
    )
    .reset_index()
    .rename(
        columns={
            "CTRP": "ctrp_drug_id",
            "PRISM": "prism_drug_id",
        }
    )
)

dual_screen_compounds = (
    dual_screen_hypotheses[
        [
            "gdsc_drug_id",
            "ctrp_drug_id",
            "prism_drug_id",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
# =============================================================================
# Quantify CTRP–PRISM and three-way overlap
# =============================================================================

def summarize_dual_screen_overlap(row):
    gdsc_models = gdsc_model_sets[
        row["gdsc_drug_id"]
    ]
    ctrp_models = ctrp_model_sets[
        int(row["ctrp_drug_id"])
    ]
    prism_models = prism_model_sets[
        row["prism_drug_id"]
    ]

    ctrp_prism_shared = ctrp_models & prism_models
    three_way_shared = (
        gdsc_models
        & ctrp_models
        & prism_models
    )

    return pd.Series(
        {
            "ctrp_prism_shared_models": len(ctrp_prism_shared),
            "ctrp_prism_shared_fraction_ctrp": (
                len(ctrp_prism_shared) / len(ctrp_models)
            ),
            "ctrp_prism_shared_fraction_prism": (
                len(ctrp_prism_shared) / len(prism_models)
            ),
            "three_way_shared_models": len(three_way_shared),
            "three_way_fraction_gdsc": (
                len(three_way_shared) / len(gdsc_models)
            ),
            "three_way_fraction_ctrp": (
                len(three_way_shared) / len(ctrp_models)
            ),
            "three_way_fraction_prism": (
                len(three_way_shared) / len(prism_models)
            ),
        }
    )


dual_screen_overlap = pd.concat(
    [
        dual_screen_compounds,
        dual_screen_compounds.apply(
            summarize_dual_screen_overlap,
            axis=1,
        ),
    ],
    axis=1,
)

replication_manifest = (
    replication_manifest
    .merge(
        dual_screen_overlap.drop(
            columns=[
                "ctrp_drug_id",
                "prism_drug_id",
            ]
        ),
        on="gdsc_drug_id",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Dual-screen evaluable hypotheses:",
    len(dual_screen_hypotheses),
)
print(
    "Unique dual-screen compounds:",
    len(dual_screen_compounds),
)

display(
    dual_screen_overlap[
        [
            "ctrp_prism_shared_models",
            "three_way_shared_models",
            "ctrp_prism_shared_fraction_ctrp",
            "ctrp_prism_shared_fraction_prism",
            "three_way_fraction_ctrp",
            "three_way_fraction_prism",
        ]
    ]
    .median()
    .rename("median")
    .to_frame()
)

In [ ]:
# =============================================================================
# Freeze and persist the notebook-603 replication hypothesis manifest
# =============================================================================

replication_hypothesis_manifest = (
    replication_manifest
    .assign(
        source_gdsc_associations=(
            "phase6.600.gdsc_program_drug_associations"
        ),
        source_compound_catalog=(
            "phase6.600.cross_resource_compound_catalog"
        ),
        source_drug_eligibility=(
            "phase6.600.drug_eligibility"
        ),
        source_gdsc_universe=(
            "phase6.600.gdsc_analysis_universe"
        ),
        source_external_universe=lambda data: np.where(
            data["external_resource"].eq("CTRP"),
            "phase6.600.ctrp_analysis_universe",
            "phase6.600.prism_analysis_universe",
        ),
        specification_date="2026-09-22",
    )
    .sort_values(
        [
            "gdsc_drug_id",
            "program",
            "external_resource",
        ]
    )
    .reset_index(drop=True)
)

# Local integrity of the new notebook-603 analytical object.
assert len(replication_hypothesis_manifest) == 676
assert not replication_hypothesis_manifest[
    [
        "gdsc_drug_id",
        "program",
        "external_resource",
    ]
].duplicated().any()

phase6_output_dir = (
    Paths.root
    / "data"
    / "processed"
    / "pharmacogenomic_contexts"
)

phase6_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

replication_manifest_path = (
    phase6_output_dir
    / "603_replication_hypothesis_manifest.parquet"
)

replication_hypothesis_manifest.to_parquet(
    replication_manifest_path,
    index=False,
)

print(
    "Frozen replication-manifest shape:",
    replication_hypothesis_manifest.shape,
)
print(
    "Primary evaluable hypotheses:",
    replication_hypothesis_manifest[
        "primary_evaluable"
    ].sum(),
)
print(
    "Non-overlap evaluable hypotheses:",
    replication_hypothesis_manifest[
        "nonoverlap_eligible"
    ].fillna(False).sum(),
)
print(
    "Manifest path:",
    project_relative_path(replication_manifest_path),
)

## External cross-screen inference

The prospective replication hypothesis manifest is now fixed before inspection of any CTRP or PRISM association result.

All subsequent primary external analyses are restricted to hypotheses labeled `EVALUABLE_PRIMARY` in this manifest. Hypotheses classified as not evaluable remain in the reporting universe but do not enter external model fitting or multiplicity correction.

For each evaluable `exact compound × consensus program` hypothesis, CTRP and PRISM are analyzed separately using the frozen lineage-adjusted model:

`response ~ program_score + C(OncotreeLineage)`

with HC3 heteroskedasticity-robust standard errors and two-sided inference.

Benjamini–Hochberg correction is applied separately to the complete evaluable CTRP and PRISM hypothesis families defined by the frozen manifest.

A screen-level result is classified as replicated only when:

1. `q_external < 0.05`; and
2. the external coefficient has the same direction as the frozen GDSC coefficient.

Native coefficients remain the primary within-screen effect representation. Standardized coefficients are calculated only for descriptive cross-screen comparison and do not define replication.

The prespecified non-overlapping-model analysis is secondary and uses its own screen-specific multiplicity families. It may strengthen a primary replicated result but cannot rescue a primary full-screen result that does not satisfy the frozen replication rule.


In [ ]:
# =============================================================================
# Select primary-evaluable external compounds
# =============================================================================

PROGRAM_COLUMNS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

score_handoff = program_scores[
    [
        "ModelID",
        *PROGRAM_COLUMNS,
        "score_origin",
    ]
]

ctrp_primary_drug_ids = (
    replication_hypothesis_manifest.loc[
        replication_hypothesis_manifest["primary_evaluable"]
        & replication_hypothesis_manifest[
            "external_resource"
        ].eq("CTRP"),
        "external_drug_id",
    ]
    .drop_duplicates()
    .astype(int)
    .tolist()
)

prism_primary_drug_ids = (
    replication_hypothesis_manifest.loc[
        replication_hypothesis_manifest["primary_evaluable"]
        & replication_hypothesis_manifest[
            "external_resource"
        ].eq("PRISM"),
        "external_drug_id",
    ]
    .drop_duplicates()
    .tolist()
)

In [ ]:
# =============================================================================
# Prepare primary external modeling universes with frozen program scores
# =============================================================================

ctrp_primary_modeling = (
    ctrp_universe.loc[
        ctrp_universe["master_cpd_id"].isin(
            ctrp_primary_drug_ids
        )
    ]
    .merge(
        score_handoff,
        on="ModelID",
        how="left",
        validate="many_to_one",
    )
)

prism_primary_modeling = (
    prism_universe.loc[
        prism_universe["broad_id"].isin(
            prism_primary_drug_ids
        )
    ]
    .merge(
        score_handoff,
        on="ModelID",
        how="left",
        validate="many_to_one",
    )
)

assert not ctrp_primary_modeling[
    PROGRAM_COLUMNS
].isna().any().any()

assert not prism_primary_modeling[
    PROGRAM_COLUMNS
].isna().any().any()

print(
    "CTRP primary modeling universe shape:",
    ctrp_primary_modeling.shape,
)
print(
    "CTRP compounds:",
    ctrp_primary_modeling[
        "master_cpd_id"
    ].nunique(),
)
print(
    "PRISM primary modeling universe shape:",
    prism_primary_modeling.shape,
)
print(
    "PRISM compounds:",
    prism_primary_modeling[
        "broad_id"
    ].nunique(),
)

print(
    "CTRP score origins:",
    ctrp_primary_modeling[
        "score_origin"
    ].value_counts().to_dict(),
)
print(
    "PRISM score origins:",
    prism_primary_modeling[
        "score_origin"
    ].value_counts().to_dict(),
)

In [ ]:
# =============================================================================
# Helpers for notebook-603 external association fits
# =============================================================================

def get_external_drug_data(row):
    if row.external_resource == "CTRP":
        return ctrp_primary_modeling.loc[
            ctrp_primary_modeling["master_cpd_id"].eq(
                int(row.external_drug_id)
            )
        ].copy()

    return prism_primary_modeling.loc[
        prism_primary_modeling["broad_id"].eq(
            row.external_drug_id
        )
    ].copy()


def restrict_supported_lineages(data):
    lineage_counts = (
        data.groupby("OncotreeLineage")
        .size()
    )
    supported = lineage_counts.loc[
        lineage_counts >= 20
    ].index

    return data.loc[
        data["OncotreeLineage"].isin(supported)
    ].copy()


def fit_lineage_adjusted_association(data, program):
    model = ols(
        formula=(
            f"response_value ~ "
            f"{program} + C(OncotreeLineage)"
        ),
        data=data,
    ).fit(
        cov_type="HC3",
        use_t=True,
    )

    confidence_interval = model.conf_int(
        alpha=0.05
    ).loc[program]

    beta = model.params[program]

    return {
        "n_models": len(data),
        "n_lineages": data["OncotreeLineage"].nunique(),
        "beta": beta,
        "se_hc3": model.bse[program],
        "ci95_low": confidence_interval.iloc[0],
        "ci95_high": confidence_interval.iloc[1],
        "t_value": model.tvalues[program],
        "p_value": model.pvalues[program],
        "df_resid": model.df_resid,
        "beta_standardized": (
            beta
            * data[program].std()
            / data["response_value"].std()
        ),
    }

In [ ]:
# =============================================================================
# Fit the frozen primary external association families
# =============================================================================

primary_external_rows = []

for row in replication_hypothesis_manifest.loc[
    replication_hypothesis_manifest["primary_evaluable"]
].itertuples(index=False):

    drug_data = get_external_drug_data(row)
    fit = fit_lineage_adjusted_association(
        drug_data,
        row.program,
    )

    primary_external_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "drug_name_key": row.drug_name_key,
            "external_resource": row.external_resource,
            "external_drug_id": row.external_drug_id,
            "primary_screen": row.primary_screen,
            "program": row.program,
            "response_metric": (
                "area_under_curve"
                if row.external_resource == "CTRP"
                else "auc"
            ),
            **fit,
            "gdsc_beta": row.gdsc_beta,
            "gdsc_direction": row.gdsc_direction,
        }
    )

primary_external_associations = (
    pd.DataFrame(primary_external_rows)
    .sort_values(
        [
            "external_resource",
            "gdsc_drug_id",
            "program",
        ]
    )
    .reset_index(drop=True)
)

assert len(primary_external_associations) == 177
assert primary_external_associations[
    "p_value"
].notna().all()

assert np.isfinite(
    primary_external_associations["p_value"]
).all()

display(
    primary_external_associations.groupby(
        "external_resource"
    )
    .agg(
        tests=("program", "size"),
        compounds=("external_drug_id", "nunique"),
        median_models=("n_models", "median"),
        median_lineages=("n_lineages", "median"),
    )
    .reset_index()
)

print(
    "Primary external tests fitted:",
    len(primary_external_associations),
)

In [ ]:
# =============================================================================
# Apply screen-specific FDR and assign primary replication status
# =============================================================================

primary_external_associations["q_value"] = np.nan
primary_external_associations["fdr_05"] = False

for resource in ["CTRP", "PRISM"]:
    resource_mask = (
        primary_external_associations[
            "external_resource"
        ].eq(resource)
    )

    reject_fdr, q_values, _, _ = multipletests(
        primary_external_associations.loc[
            resource_mask,
            "p_value",
        ],
        alpha=0.05,
        method="fdr_bh",
    )

    primary_external_associations.loc[
        resource_mask,
        "q_value",
    ] = q_values

    primary_external_associations.loc[
        resource_mask,
        "fdr_05",
    ] = reject_fdr


primary_external_associations[
    "external_direction"
] = np.where(
    primary_external_associations["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

primary_external_associations[
    "direction_concordant"
] = (
    primary_external_associations[
        "external_direction"
    ]
    == primary_external_associations[
        "gdsc_direction"
    ]
)

primary_external_associations[
    "replication_status"
] = np.select(
    [
        primary_external_associations["fdr_05"]
        & primary_external_associations[
            "direction_concordant"
        ],
        primary_external_associations[
            "direction_concordant"
        ],
    ],
    [
        "SCREEN_REPLICATED",
        "TESTED_DIRECTION_CONCORDANT_NOT_FDR",
    ],
    default="TESTED_DIRECTION_DISCORDANT",
)

display(
    primary_external_associations.groupby(
        [
            "external_resource",
            "replication_status",
        ],
        dropna=False,
    )
    .size()
    .rename("hypotheses")
    .reset_index()
)

display(
    primary_external_associations.groupby(
        "external_resource"
    )
    .agg(
        tests=("program", "size"),
        fdr_significant=("fdr_05", "sum"),
        direction_concordant=(
            "direction_concordant",
            "sum",
        ),
        screen_replicated=(
            "replication_status",
            lambda values: (
                values == "SCREEN_REPLICATED"
            ).sum(),
        ),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Fit the prespecified non-overlapping-model association families
# =============================================================================

nonoverlap_external_rows = []

for row in replication_hypothesis_manifest.loc[
    replication_hypothesis_manifest["nonoverlap_eligible"].eq(True)
].itertuples(index=False):

    drug_data = get_external_drug_data(row)
    drug_data = drug_data.loc[
        ~drug_data["ModelID"].isin(
            gdsc_model_sets[row.gdsc_drug_id]
        )
    ]
    drug_data = restrict_supported_lineages(
        drug_data
    )

    fit = fit_lineage_adjusted_association(
        drug_data,
        row.program,
    )

    nonoverlap_external_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "drug_name_key": row.drug_name_key,
            "external_resource": row.external_resource,
            "external_drug_id": row.external_drug_id,
            "primary_screen": row.primary_screen,
            "program": row.program,
            "response_metric": (
                "area_under_curve"
                if row.external_resource == "CTRP"
                else "auc"
            ),
            **fit,
            "gdsc_beta": row.gdsc_beta,
            "gdsc_direction": row.gdsc_direction,
        }
    )

nonoverlap_external_associations = (
    pd.DataFrame(nonoverlap_external_rows)
    .sort_values(
        [
            "external_resource",
            "gdsc_drug_id",
            "program",
        ]
    )
    .reset_index(drop=True)
)

assert len(
    nonoverlap_external_associations
) == 76

assert (
    nonoverlap_external_associations["n_models"]
    >= 100
).all()

assert (
    nonoverlap_external_associations["n_lineages"]
    >= 3
).all()

assert nonoverlap_external_associations[
    "p_value"
].notna().all()

display(
    nonoverlap_external_associations.groupby(
        "external_resource"
    )
    .agg(
        tests=("program", "size"),
        compounds=("external_drug_id", "nunique"),
        median_models=("n_models", "median"),
        median_lineages=("n_lineages", "median"),
    )
    .reset_index()
)

print(
    "Non-overlap external tests fitted:",
    len(nonoverlap_external_associations),
)

In [ ]:
# =============================================================================
# Apply screen-specific FDR to non-overlap analyses
# =============================================================================

nonoverlap_external_associations["q_value"] = np.nan
nonoverlap_external_associations["fdr_05"] = False

for resource in ["CTRP", "PRISM"]:
    resource_mask = (
        nonoverlap_external_associations[
            "external_resource"
        ].eq(resource)
    )

    reject_fdr, q_values, _, _ = multipletests(
        nonoverlap_external_associations.loc[
            resource_mask,
            "p_value",
        ],
        alpha=0.05,
        method="fdr_bh",
    )

    nonoverlap_external_associations.loc[
        resource_mask,
        "q_value",
    ] = q_values

    nonoverlap_external_associations.loc[
        resource_mask,
        "fdr_05",
    ] = reject_fdr

nonoverlap_external_associations[
    "external_direction"
] = np.where(
    nonoverlap_external_associations["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

nonoverlap_external_associations[
    "direction_concordant"
] = (
    nonoverlap_external_associations[
        "external_direction"
    ]
    == nonoverlap_external_associations[
        "gdsc_direction"
    ]
)

In [ ]:
# =============================================================================
# Assess non-overlap corroboration without rescue
# =============================================================================

primary_status = (
    primary_external_associations[
        [
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
            "program",
            "replication_status",
        ]
    ]
    .rename(
        columns={
            "replication_status":
                "primary_replication_status",
        }
    )
)

nonoverlap_external_associations = (
    nonoverlap_external_associations
    .merge(
        primary_status,
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

nonoverlap_external_associations[
    "nonoverlap_favorable"
] = (
    nonoverlap_external_associations["fdr_05"]
    & nonoverlap_external_associations[
        "direction_concordant"
    ]
)

nonoverlap_external_associations[
    "nonoverlap_model_corroboration"
] = (
    nonoverlap_external_associations[
        "primary_replication_status"
    ].eq("SCREEN_REPLICATED")
    & nonoverlap_external_associations[
        "nonoverlap_favorable"
    ]
)

nonoverlap_external_associations[
    "nonoverlap_support_label"
] = np.where(
    nonoverlap_external_associations[
        "nonoverlap_model_corroboration"
    ],
    "NONOVERLAP_MODEL_CORROBORATION",
    pd.NA,
)

display(
    nonoverlap_external_associations.groupby(
        "external_resource"
    )
    .agg(
        tests=("program", "size"),
        fdr_significant=("fdr_05", "sum"),
        direction_concordant=("direction_concordant", "sum"),
        favorable_nonoverlap=("nonoverlap_favorable", "sum"),
        primary_replicated=(
            "primary_replication_status",
            lambda values: (
                values == "SCREEN_REPLICATED"
            ).sum(),
        ),
        corroborated=("nonoverlap_model_corroboration", "sum"),
    )
    .reset_index()
)

display(
    nonoverlap_external_associations.loc[
        nonoverlap_external_associations[
            "nonoverlap_favorable"
        ]
        & ~nonoverlap_external_associations[
            "nonoverlap_model_corroboration"
        ],
        [
            "external_resource",
            "gdsc_drug_id",
            "program",
            "primary_replication_status",
            "q_value",
            "direction_concordant",
        ],
    ]
)

In [ ]:
# =============================================================================
# Helpers for overlap-manifest provenance
# =============================================================================

score_origin_by_model = (
    program_scores
    .set_index("ModelID")["score_origin"]
)


def serialize_sorted(values):
    return json.dumps(
        sorted(values),
        separators=(",", ":"),
    )


def score_origin_fields(prefix, model_ids):
    origins = score_origin_by_model.loc[
        list(model_ids)
    ]

    frozen = origins.eq(
        "frozen_phase4"
    ).sum()
    projected = origins.eq(
        "projected_from_frozen_phase4"
    ).sum()
    total = len(origins)

    return {
        f"{prefix}_frozen_phase4_models": frozen,
        f"{prefix}_projected_models": projected,
        f"{prefix}_projected_fraction": (
            projected / total
            if total > 0
            else np.nan
        ),
    }

In [ ]:
# =============================================================================
# Build the compound-resource overlap scaffold
# =============================================================================

overlap_columns = [
    "gdsc_drug_id",
    "DRUG_NAME",
    "drug_name_key",
    "external_resource",
    "external_drug_id",
    "primary_screen",
    "preanalysis_status",
    "primary_evaluable",
    "frozen_supported_lineages",
    "frozen_supported_models",
    "gdsc_primary_models",
    "external_primary_models",
    "shared_models",
    "external_nonoverlap_models",
    "shared_fraction_external",
    "nonoverlap_supported_lineages",
    "nonoverlap_supported_models",
    "nonoverlap_eligible",
    "nonoverlap_status",
    "ctrp_prism_shared_models",
    "ctrp_prism_shared_fraction_ctrp",
    "ctrp_prism_shared_fraction_prism",
    "three_way_shared_models",
    "three_way_fraction_gdsc",
    "three_way_fraction_ctrp",
    "three_way_fraction_prism",
]

overlap_compounds = (
    replication_hypothesis_manifest[
        overlap_columns
    ]
    .drop_duplicates(
        subset=[
            "gdsc_drug_id",
            "external_resource",
        ]
    )
    .sort_values(
        [
            "gdsc_drug_id",
            "external_resource",
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# =============================================================================
# Attach deterministic model-set and lineage provenance
# =============================================================================

def summarize_overlap_provenance(row):
    if not row["primary_evaluable"]:
        return pd.Series(
            {
                "gdsc_model_ids": pd.NA,
                "external_model_ids": pd.NA,
                "shared_model_ids": pd.NA,
                "external_nonoverlap_model_ids": pd.NA,
                "shared_lineage_ids": pd.NA,
            }
        )

    gdsc_data = gdsc_universe.loc[
        gdsc_universe["DRUG_ID"].eq(
            row["gdsc_drug_id"]
        )
    ]

    if row["external_resource"] == "CTRP":
        external_data = ctrp_primary_modeling.loc[
            ctrp_primary_modeling[
                "master_cpd_id"
            ].eq(
                int(row["external_drug_id"])
            )
        ]
    else:
        external_data = prism_primary_modeling.loc[
            prism_primary_modeling[
                "broad_id"
            ].eq(
                row["external_drug_id"]
            )
        ]

    gdsc_models = set(gdsc_data["ModelID"])
    external_models = set(external_data["ModelID"])
    shared_models = gdsc_models & external_models
    nonoverlap_models = external_models - gdsc_models
    shared_lineages = (
        set(gdsc_data["OncotreeLineage"])
        & set(external_data["OncotreeLineage"])
    )

    return pd.Series(
        {
            "gdsc_model_ids": serialize_sorted(gdsc_models),
            "external_model_ids": serialize_sorted(external_models),
            "shared_model_ids": serialize_sorted(shared_models),
            "external_nonoverlap_model_ids": serialize_sorted(
                nonoverlap_models
            ),
            "shared_lineage_ids": serialize_sorted(shared_lineages),
            "gdsc_lineages": gdsc_data[
                "OncotreeLineage"
            ].nunique(),
            "external_lineages": external_data[
                "OncotreeLineage"
            ].nunique(),
            "shared_lineages": len(shared_lineages),
            "external_nonoverlap_lineages": (
                external_data.loc[
                    external_data["ModelID"].isin(
                        nonoverlap_models
                    ),
                    "OncotreeLineage",
                ].nunique()
            ),
            **score_origin_fields(
                "gdsc",
                gdsc_models,
            ),
            **score_origin_fields(
                "external",
                external_models,
            ),
            **score_origin_fields(
                "shared",
                shared_models,
            ),
            **score_origin_fields(
                "nonoverlap",
                nonoverlap_models,
            ),
        }
    )


overlap_provenance = overlap_compounds.apply(
    summarize_overlap_provenance,
    axis=1,
)

cell_line_overlap_manifest = pd.concat(
    [
        overlap_compounds,
        overlap_provenance,
    ],
    axis=1,
)

assert not cell_line_overlap_manifest[
    [
        "gdsc_drug_id",
        "external_resource",
    ]
].duplicated().any()

In [ ]:
# =============================================================================
# Persist the notebook-603 cell-line overlap manifest
# =============================================================================

cell_line_overlap_path = (
    phase6_output_dir
    / "603_cell_line_overlap_manifest.parquet"
)

cell_line_overlap_manifest.to_parquet(
    cell_line_overlap_path,
    index=False,
)

print(
    "Cell-line overlap manifest shape:",
    cell_line_overlap_manifest.shape,
)
print(
    "Primary-evaluable compound-resource pairs:",
    cell_line_overlap_manifest[
        "primary_evaluable"
    ].sum(),
)
print(
    "Overlap manifest path:",
    project_relative_path(
        cell_line_overlap_path
    ),
)

display(
    cell_line_overlap_manifest.loc[
        cell_line_overlap_manifest[
            "primary_evaluable"
        ]
    ]
    .groupby("external_resource")
    .agg(
        compound_resource_pairs=("gdsc_drug_id", "size"),
        median_shared_fraction=("shared_fraction_external", "median"),
        median_external_projected_fraction=(
            "external_projected_fraction",
            "median",
        ),
        median_nonoverlap_projected_fraction=(
            "nonoverlap_projected_fraction",
            "median",
        ),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Helper for the frozen 20 / 3 / 100 coverage rule
# =============================================================================

def summarize_coverage(data):
    supported_data = restrict_supported_lineages(
        data
    )

    return {
        "supported_lineages": (
            supported_data[
                "OncotreeLineage"
            ].nunique()
        ),
        "supported_models": len(supported_data),
        "eligible": (
            supported_data[
                "OncotreeLineage"
            ].nunique() >= 3
            and len(supported_data) >= 100
        ),
    }

In [ ]:
# =============================================================================
# Determine shared-only diagnostic coverage
# =============================================================================

shared_only_rows = []

for row in cell_line_overlap_manifest.loc[
    cell_line_overlap_manifest["primary_evaluable"]
].itertuples(index=False):

    gdsc_models = gdsc_model_sets[row.gdsc_drug_id]

    if row.external_resource == "CTRP":
        external_data = ctrp_primary_modeling.loc[
            ctrp_primary_modeling["master_cpd_id"].eq(
                int(row.external_drug_id)
            )
        ]
    else:
        external_data = prism_primary_modeling.loc[
            prism_primary_modeling["broad_id"].eq(
                row.external_drug_id
            )
        ]

    shared_data = external_data.loc[
        external_data["ModelID"].isin(gdsc_models)
    ]

    coverage = summarize_coverage(shared_data)

    shared_only_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "external_resource": row.external_resource,
            "external_drug_id": row.external_drug_id,
            "shared_supported_lineages": coverage["supported_lineages"],
            "shared_supported_models": coverage["supported_models"],
            "shared_only_eligible": coverage["eligible"],
        }
    )

shared_only_coverage = pd.DataFrame(shared_only_rows)

display(
    shared_only_coverage.groupby("external_resource")
    .agg(
        compound_resource_pairs=("gdsc_drug_id", "size"),
        shared_only_evaluable=("shared_only_eligible", "sum"),
        median_supported_models=("shared_supported_models", "median"),
        median_supported_lineages=("shared_supported_lineages", "median"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Fit shared-only diagnostic associations
# =============================================================================

shared_only_hypotheses = (
    replication_hypothesis_manifest.loc[
        replication_hypothesis_manifest["primary_evaluable"]
    ]
    .merge(
        shared_only_coverage,
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
        ],
        how="left",
        validate="many_to_one",
    )
)

shared_only_rows = []

for row in shared_only_hypotheses.loc[
    shared_only_hypotheses["shared_only_eligible"]
].itertuples(index=False):

    data = get_external_drug_data(row)
    data = data.loc[
        data["ModelID"].isin(
            gdsc_model_sets[row.gdsc_drug_id]
        )
    ]
    data = restrict_supported_lineages(
        data
    )

    fit = fit_lineage_adjusted_association(
        data,
        row.program,
    )

    shared_only_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "external_resource": row.external_resource,
            "external_drug_id": row.external_drug_id,
            "program": row.program,
            "n_models": fit["n_models"],
            "n_lineages": fit["n_lineages"],
            "beta": fit["beta"],
            "beta_standardized": fit[
                "beta_standardized"
            ],
        }
    )

shared_only_associations = pd.DataFrame(
    shared_only_rows
)

display(
    shared_only_associations.groupby(
        "external_resource"
    )
    .agg(
        diagnostic_tests=("program", "size"),
        compounds=("external_drug_id", "nunique"),
        median_models=("n_models", "median"),
        median_lineages=("n_lineages", "median"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Compare shared-only diagnostic directions
# =============================================================================

shared_only_associations["shared_direction"] = np.where(
    shared_only_associations["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

shared_only_associations = (
    shared_only_associations
    .merge(
        primary_external_associations[
            [
                "gdsc_drug_id",
                "external_resource",
                "external_drug_id",
                "program",
                "gdsc_direction",
                "external_direction",
                "replication_status",
            ]
        ],
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

shared_only_associations["matches_gdsc_direction"] = (
    shared_only_associations["shared_direction"]
    == shared_only_associations["gdsc_direction"]
)

shared_only_associations["matches_primary_direction"] = (
    shared_only_associations["shared_direction"]
    == shared_only_associations["external_direction"]
)

display(
    shared_only_associations.groupby("external_resource")
    .agg(
        diagnostic_tests=("program", "size"),
        matches_gdsc=("matches_gdsc_direction", "sum"),
        matches_primary=("matches_primary_direction", "sum"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Determine shared-lineage diagnostic coverage
# =============================================================================

shared_lineage_rows = []

for row in cell_line_overlap_manifest.loc[
    cell_line_overlap_manifest["primary_evaluable"]
].itertuples(index=False):

    gdsc_data = gdsc_universe.loc[
        gdsc_universe["DRUG_ID"].eq(row.gdsc_drug_id)
    ]

    if row.external_resource == "CTRP":
        external_data = ctrp_primary_modeling.loc[
            ctrp_primary_modeling["master_cpd_id"].eq(
                int(row.external_drug_id)
            )
        ]
    else:
        external_data = prism_primary_modeling.loc[
            prism_primary_modeling["broad_id"].eq(
                row.external_drug_id
            )
        ]

    shared_lineages = (
        set(gdsc_data["OncotreeLineage"])
        & set(external_data["OncotreeLineage"])
    )

    diagnostic_data = external_data.loc[
        external_data["OncotreeLineage"].isin(
            shared_lineages
        )
    ]

    coverage = summarize_coverage(diagnostic_data)

    shared_lineage_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "external_resource": row.external_resource,
            "external_drug_id": row.external_drug_id,
            "shared_lineage_count": len(shared_lineages),
            "supported_lineages": coverage["supported_lineages"],
            "supported_models": coverage["supported_models"],
            "shared_lineage_eligible": coverage["eligible"],
        }
    )

shared_lineage_coverage = pd.DataFrame(
    shared_lineage_rows
)

display(
    shared_lineage_coverage.groupby("external_resource")
    .agg(
        compound_resource_pairs=("gdsc_drug_id", "size"),
        evaluable=("shared_lineage_eligible", "sum"),
        median_shared_lineages=("shared_lineage_count", "median"),
        median_supported_models=("supported_models", "median"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Fit shared-lineage diagnostic associations
# =============================================================================

shared_lineage_rows = []

for row in replication_hypothesis_manifest.loc[
    replication_hypothesis_manifest["primary_evaluable"]
].itertuples(index=False):

    gdsc_lineages = set(
        gdsc_universe.loc[
            gdsc_universe["DRUG_ID"].eq(
                row.gdsc_drug_id
            ),
            "OncotreeLineage",
        ]
    )

    data = get_external_drug_data(row)
    data = data.loc[
        data["OncotreeLineage"].isin(
            gdsc_lineages
        )
    ]
    data = restrict_supported_lineages(
        data
    )

    fit = fit_lineage_adjusted_association(
        data,
        row.program,
    )

    shared_lineage_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "external_resource": row.external_resource,
            "external_drug_id": row.external_drug_id,
            "program": row.program,
            "n_models": fit["n_models"],
            "n_lineages": fit["n_lineages"],
            "beta": fit["beta"],
            "beta_standardized": fit[
                "beta_standardized"
            ],
        }
    )

shared_lineage_associations = pd.DataFrame(
    shared_lineage_rows
)

display(
    shared_lineage_associations.groupby(
        "external_resource"
    )
    .agg(
        diagnostic_tests=("program", "size"),
        median_models=("n_models", "median"),
        median_lineages=("n_lineages", "median"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Compare shared-lineage diagnostic directions
# =============================================================================

shared_lineage_associations["shared_lineage_direction"] = np.where(
    shared_lineage_associations["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

shared_lineage_associations = (
    shared_lineage_associations
    .merge(
        primary_external_associations[
            [
                "gdsc_drug_id",
                "external_resource",
                "external_drug_id",
                "program",
                "gdsc_direction",
                "external_direction",
            ]
        ],
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

shared_lineage_associations["matches_gdsc_direction"] = (
    shared_lineage_associations["shared_lineage_direction"]
    == shared_lineage_associations["gdsc_direction"]
)

shared_lineage_associations["matches_primary_direction"] = (
    shared_lineage_associations["shared_lineage_direction"]
    == shared_lineage_associations["external_direction"]
)

display(
    shared_lineage_associations.groupby("external_resource")
    .agg(
        diagnostic_tests=("program", "size"),
        matches_gdsc=("matches_gdsc_direction", "sum"),
        matches_primary=("matches_primary_direction", "sum"),
    )
    .reset_index()
)

In [ ]:
# =============================================================================
# Determine CTRP single-experiment sensitivity coverage
# =============================================================================

ctrp_sensitivity_rows = []

ctrp_pairs = cell_line_overlap_manifest.loc[
    cell_line_overlap_manifest["primary_evaluable"]
    & cell_line_overlap_manifest["external_resource"].eq("CTRP")
]

for row in ctrp_pairs.itertuples(index=False):
    data = ctrp_primary_modeling.loc[
        ctrp_primary_modeling["master_cpd_id"].eq(
            int(row.external_drug_id)
        )
        & ctrp_primary_modeling["n_experiments"].eq(1)
    ]

    coverage = summarize_coverage(data)

    ctrp_sensitivity_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "external_drug_id": row.external_drug_id,
            "supported_lineages": coverage["supported_lineages"],
            "supported_models": coverage["supported_models"],
            "single_experiment_eligible": coverage["eligible"],
        }
    )

ctrp_single_experiment_coverage = pd.DataFrame(
    ctrp_sensitivity_rows
)

display(
    ctrp_single_experiment_coverage.agg(
        compound_resource_pairs=("gdsc_drug_id", "size"),
        evaluable=("single_experiment_eligible", "sum"),
        median_supported_models=("supported_models", "median"),
        median_supported_lineages=("supported_lineages", "median"),
    )
)

In [ ]:
# =============================================================================
# Fit CTRP single-experiment sensitivity associations
# =============================================================================

ctrp_single_experiment_rows = []

ctrp_hypotheses = replication_hypothesis_manifest.loc[
    replication_hypothesis_manifest["primary_evaluable"]
    & replication_hypothesis_manifest[
        "external_resource"
    ].eq("CTRP")
]

for row in ctrp_hypotheses.itertuples(index=False):
    data = get_external_drug_data(row)
    data = data.loc[
        data["n_experiments"].eq(1)
    ]
    data = restrict_supported_lineages(
        data
    )

    fit = fit_lineage_adjusted_association(
        data,
        row.program,
    )

    ctrp_single_experiment_rows.append(
        {
            "gdsc_drug_id": row.gdsc_drug_id,
            "external_drug_id": row.external_drug_id,
            "program": row.program,
            "n_models": fit["n_models"],
            "n_lineages": fit["n_lineages"],
            "beta": fit["beta"],
            "beta_standardized": fit[
                "beta_standardized"
            ],
        }
    )

ctrp_single_experiment_associations = (
    pd.DataFrame(
        ctrp_single_experiment_rows
    )
)

print(
    "CTRP single-experiment sensitivity tests:",
    len(ctrp_single_experiment_associations),
)

display(
    ctrp_single_experiment_associations[
        [
            "n_models",
            "n_lineages",
            "beta_standardized",
        ]
    ].median()
)

In [ ]:
# =============================================================================
# Compare CTRP single-experiment sensitivity directions
# =============================================================================

ctrp_single_experiment_associations[
    "sensitivity_direction"
] = np.where(
    ctrp_single_experiment_associations["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

ctrp_single_experiment_associations = (
    ctrp_single_experiment_associations
    .merge(
        primary_external_associations.loc[
            primary_external_associations[
                "external_resource"
            ].eq("CTRP"),
            [
                "gdsc_drug_id",
                "external_drug_id",
                "program",
                "gdsc_direction",
                "external_direction",
            ],
        ],
        on=[
            "gdsc_drug_id",
            "external_drug_id",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

ctrp_single_experiment_associations[
    "matches_gdsc_direction"
] = (
    ctrp_single_experiment_associations[
        "sensitivity_direction"
    ]
    == ctrp_single_experiment_associations[
        "gdsc_direction"
    ]
)

ctrp_single_experiment_associations[
    "matches_primary_direction"
] = (
    ctrp_single_experiment_associations[
        "sensitivity_direction"
    ]
    == ctrp_single_experiment_associations[
        "external_direction"
    ]
)

display(
    ctrp_single_experiment_associations.agg(
        diagnostic_tests=("program", "size"),
        matches_gdsc=("matches_gdsc_direction", "sum"),
        matches_primary=("matches_primary_direction", "sum"),
    )
)

In [ ]:
# =============================================================================
# Identify dual external-screen replication support
# =============================================================================

dual_screen_status = (
    primary_external_associations
    .pivot(
        index=[
            "gdsc_drug_id",
            "program",
        ],
        columns="external_resource",
        values="replication_status",
    )
    .dropna(
        subset=[
            "CTRP",
            "PRISM",
        ]
    )
    .reset_index()
)

dual_screen_status[
    "dual_external_status"
] = np.where(
    dual_screen_status["CTRP"].eq(
        "SCREEN_REPLICATED"
    )
    & dual_screen_status["PRISM"].eq(
        "SCREEN_REPLICATED"
    ),
    "REPLICATED_IN_BOTH_EXTERNAL_SCREENS",
    pd.NA,
)

print(
    "Dual-screen evaluable hypotheses:",
    len(dual_screen_status),
)
print(
    "Replicated in both external screens:",
    dual_screen_status[
        "dual_external_status"
    ].notna().sum(),
)

In [ ]:
# =============================================================================
# Assemble frozen notebook-603 replication summary
# =============================================================================

replication_summary = (
    replication_hypothesis_manifest
    .merge(
        primary_external_associations[
            [
                "gdsc_drug_id",
                "external_resource",
                "external_drug_id",
                "program",
                "beta",
                "q_value",
                "external_direction",
                "replication_status",
            ]
        ],
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

replication_summary["final_screen_status"] = np.where(
    replication_summary["primary_evaluable"],
    replication_summary["replication_status"],
    replication_summary["preanalysis_status"],
)

replication_summary = (
    replication_summary
    .merge(
        nonoverlap_external_associations[
            [
                "gdsc_drug_id",
                "external_resource",
                "external_drug_id",
                "program",
                "nonoverlap_support_label",
            ]
        ],
        on=[
            "gdsc_drug_id",
            "external_resource",
            "external_drug_id",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

replication_summary = (
    replication_summary
    .merge(
        dual_screen_status[
            [
                "gdsc_drug_id",
                "program",
                "dual_external_status",
            ]
        ],
        on=[
            "gdsc_drug_id",
            "program",
        ],
        how="left",
        validate="many_to_one",
    )
)

display(
    replication_summary.groupby(
        [
            "external_resource",
            "final_screen_status",
        ],
        dropna=False,
    )
    .size()
    .rename("hypotheses")
    .reset_index()
)

In [ ]:
# =============================================================================
# Persist frozen notebook-603 replication results
# =============================================================================

primary_external_path = (
    phase6_output_dir
    / "603_primary_external_associations.parquet"
)

nonoverlap_external_path = (
    phase6_output_dir
    / "603_nonoverlap_external_associations.parquet"
)

replication_summary_path = (
    phase6_output_dir
    / "603_replication_summary.parquet"
)

primary_external_associations.to_parquet(
    primary_external_path,
    index=False,
)

nonoverlap_external_associations.to_parquet(
    nonoverlap_external_path,
    index=False,
)

replication_summary.to_parquet(
    replication_summary_path,
    index=False,
)

print(
    "Primary associations:",
    project_relative_path(primary_external_path),
)
print(
    "Non-overlap associations:",
    project_relative_path(nonoverlap_external_path),
)
print(
    "Replication summary:",
    project_relative_path(replication_summary_path),
)

In [ ]:
# =============================================================================
# Load frozen notebook-601 and notebook-602 contextual handoffs
# =============================================================================

context_artifact_ids = {
    "predictive": "phase6.601.drug_level_results",
    "attribution": "phase6.602.drug_program_attribution_summary",
}

context_paths = {
    name: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for name, artifact_id in context_artifact_ids.items()
}

predictive_context = pd.read_csv(
    context_paths["predictive"]
)

attribution_context = pd.read_csv(
    context_paths["attribution"]
)

print(
    "Notebook-601 drug context:",
    predictive_context.shape,
)
print(
    "Notebook-602 drug-program context:",
    attribution_context.shape,
)

print(
    "601 SHAP-eligible drugs:",
    predictive_context["shap_eligible"].sum(),
)
print(
    "602 represented drugs:",
    attribution_context["DRUG_ID"].nunique(),
)

In [ ]:
# =============================================================================
# Append frozen notebook-601 and notebook-602 context
# =============================================================================

predictive_context_compact = (
    predictive_context[
        [
            "DRUG_ID",
            "median_r2_program",
            "median_delta_r2",
            "positive_delta_repeats",
            "shap_eligible",
        ]
    ]
    .rename(columns={"DRUG_ID": "gdsc_drug_id"})
)

attribution_context_compact = (
    attribution_context[
        [
            "DRUG_ID",
            "program",
            "median_repeat_mean_abs_shap",
            "dominant_repeat_fraction",
            "median_coefficient",
            "fraction_positive",
            "fraction_negative",
        ]
    ]
    .rename(columns={"DRUG_ID": "gdsc_drug_id"})
)

replication_summary_contextual = (
    replication_summary
    .merge(
        predictive_context_compact,
        on="gdsc_drug_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        attribution_context_compact,
        on=["gdsc_drug_id", "program"],
        how="left",
        validate="many_to_one",
    )
)

print(
    "Contextual summary shape:",
    replication_summary_contextual.shape,
)
print(
    "Rows with notebook-602 attribution:",
    replication_summary_contextual[
        "median_repeat_mean_abs_shap"
    ].notna().sum(),
)

assert (
    replication_summary_contextual["final_screen_status"]
    .equals(replication_summary["final_screen_status"])
)

In [ ]:
# =============================================================================
# Persist the final contextual replication summary
# =============================================================================

replication_summary_final = (
    replication_summary_contextual
    .sort_values(
        [
            "gdsc_drug_id",
            "program",
            "external_resource",
        ]
    )
    .reset_index(drop=True)
)

replication_summary_final.to_parquet(
    replication_summary_path,
    index=False,
)

print(
    "Final replication summary shape:",
    replication_summary_final.shape,
)
print(
    "Screen-replicated rows:",
    replication_summary_final[
        "final_screen_status"
    ].eq("SCREEN_REPLICATED").sum(),
)
print(
    "Screen-replicated rows with 602 context:",
    replication_summary_final.loc[
        replication_summary_final[
            "final_screen_status"
        ].eq("SCREEN_REPLICATED"),
        "median_repeat_mean_abs_shap",
    ].notna().sum(),
)

In [ ]:
# =============================================================================
# Assemble notebook-603 final analysis metrics
# =============================================================================

analysis_metrics = {
    "hypotheses_total": int(
        len(replication_hypothesis_manifest)
    ),
    "primary_evaluable": int(
        replication_hypothesis_manifest[
            "primary_evaluable"
        ].sum()
    ),
    "screen_replicated": {
        resource: int(
            primary_external_associations.loc[
                primary_external_associations[
                    "external_resource"
                ].eq(resource),
                "replication_status",
            ].eq("SCREEN_REPLICATED").sum()
        )
        for resource in ["CTRP", "PRISM"]
    },
    "nonoverlap_corroborated": int(
        nonoverlap_external_associations[
            "nonoverlap_model_corroboration"
        ].sum()
    ),
    "replicated_both_external_screens": int(
        dual_screen_status[
            "dual_external_status"
        ].notna().sum()
    ),
    "replicated_with_602_context": int(
        replication_summary_final.loc[
            replication_summary_final[
                "final_screen_status"
            ].eq("SCREEN_REPLICATED"),
            "median_repeat_mean_abs_shap",
        ].notna().sum()
    ),
}

analysis_metrics

In [ ]:
# =============================================================================
# Assemble notebook-603 diagnostic summaries
# =============================================================================

diagnostic_summary = {
    "primary_family_sizes": {
        resource: int(count)
        for resource, count in (
            primary_external_associations.groupby(
                "external_resource"
            ).size().items()
        )
    },
    "nonoverlap_family_sizes": {
        resource: int(count)
        for resource, count in (
            nonoverlap_external_associations.groupby(
                "external_resource"
            ).size().items()
        )
    },
    "median_shared_fraction_external": {
        resource: float(value)
        for resource, value in (
            cell_line_overlap_manifest.loc[
                cell_line_overlap_manifest[
                    "primary_evaluable"
                ]
            ]
            .groupby("external_resource")[
                "shared_fraction_external"
            ]
            .median()
            .items()
        )
    },
    "shared_only_direction_matches_primary": {
        resource: int(value)
        for resource, value in (
            shared_only_associations.groupby(
                "external_resource"
            )["matches_primary_direction"]
            .sum()
            .items()
        )
    },
    "shared_lineage_direction_matches_primary": {
        resource: int(value)
        for resource, value in (
            shared_lineage_associations.groupby(
                "external_resource"
            )["matches_primary_direction"]
            .sum()
            .items()
        )
    },
    "ctrp_single_experiment_direction_matches_primary": int(
        ctrp_single_experiment_associations[
            "matches_primary_direction"
        ].sum()
    ),
}

diagnostic_summary

In [ ]:
# =============================================================================
# Assemble notebook-603 analysis metadata
# =============================================================================

analysis_metadata = {
    "schema_version": 1,
    "notebook": "603_cross_screen_replication",
    "analysis_role": (
        "external_cross_screen_pharmacogenomic_replication"
    ),
    "specification_date": "2026-09-22",
    "reference_resource": "GDSC",
    "external_resources": ["CTRP", "PRISM"],
    "exact_compound_rule": (
        "frozen_notebook600_one_to_one_normalized_name"
    ),
    "response_metrics": {
        "GDSC": "LN_IC50",
        "CTRP": "area_under_curve",
        "PRISM": "auc",
    },
    "primary_model": (
        "response_value ~ {program} + C(OncotreeLineage)"
    ),
    "inference": {
        "covariance": "HC3",
        "use_t": True,
        "tests": "two_sided",
        "multiplicity": (
            "BH_separately_within_each_external_screen"
        ),
    },
    "replication_rule": {
        "fdr_threshold": 0.05,
        "requires_gdsc_direction_concordance": True,
        "label": "SCREEN_REPLICATED",
    },
    "coverage_rule": {
        "minimum_models_per_supported_lineage": 20,
        "minimum_supported_lineages": 3,
        "minimum_supported_models": 100,
    },
    "secondary_analyses": {
        "nonoverlap": (
            "separate_screen_specific_BH_families"
        ),
        "shared_only": (
            "descriptive_native_and_standardized_effects"
        ),
        "shared_lineage": (
            "descriptive_native_and_standardized_effects"
        ),
        "ctrp_single_experiment": (
            "descriptive_native_and_standardized_effects"
        ),
    },
    "cross_screen_independence_claimed": False,
    "notebook601_602_selection_role": "none",
    "analysis_metrics": analysis_metrics,
    "diagnostic_summary": diagnostic_summary,
    "upstream_artifacts": list(INPUT_ARTIFACT_IDS),
    "context_artifacts": list(
        context_artifact_ids.values()
    ),
    "interpretation_limitations": [
        (
            "Cross-screen replication may include overlapping "
            "cell-line models and is not independent biological validation."
        ),
        (
            "Native GDSC, CTRP, and PRISM response scales are "
            "resource-specific and are not pooled."
        ),
        (
            "Non-overlap subsets are enriched for projected "
            "program scores and can have reduced lineage coverage."
        ),
        (
            "Residual proliferation, platform, lineage-composition, "
            "and other cell-line confounding remain possible."
        ),
        (
            "Notebook-601 predictive performance and notebook-602 "
            "SHAP attribution are contextual evidence only."
        ),
        (
            "Association and fitted-model attribution do not establish "
            "causal drug-response mechanisms or clinical predictiveness."
        ),
    ],
}

analysis_metadata

In [ ]:
# =============================================================================
# Persist notebook-603 analysis metadata
# =============================================================================

analysis_metadata_path = (
    phase6_output_dir
    / "603_analysis_metadata.json"
)

with open(
    analysis_metadata_path,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        analysis_metadata,
        handle,
        indent=2,
        sort_keys=True,
    )

print(
    "Analysis metadata:",
    project_relative_path(
        analysis_metadata_path
    ),
)

## Results and interpretation

Notebook 603 evaluated 177 prospectively eligible `GDSC hypothesis × external screen` tests derived exclusively from the frozen notebook-600 FDR-positive hypothesis set.

Under the prespecified screen-specific BH correction and directional-concordance rule:

* CTRP replicated 33 of 79 evaluable hypotheses.
* PRISM replicated 4 of 98 evaluable hypotheses.
* No hypothesis replicated simultaneously in both external screens among the 40 hypotheses evaluable in both.
* Four CTRP replications were additionally supported in the prespecified non-overlapping-cell-line analysis.
* No PRISM hypothesis received non-overlapping-model corroboration; PRISM non-overlap coverage was substantially more limited.

The descriptive shared-model, shared-lineage, and CTRP single-experiment diagnostics generally preserved primary coefficient direction and did not alter any replication classification.

Cell-line overlap was substantial across resources. In addition, non-overlapping external subsets were enriched for models whose program scores were projected from the frozen Phase 4 representation. These features limit claims of statistical or biological independence across screens.

Notebook-601 predictive performance and notebook-602 attribution were appended only after notebook-603 replication statuses were fixed. Thirty of the 37 screen-replicated rows have notebook-602 attribution context, but predictive validity and SHAP attribution were not used to select, promote, rescue, or downgrade any replication result.

These findings support a set of screen-specific cross-pharmacogenomic associations. They do not establish independent biological validation, causal drug-response mechanisms, therapeutic efficacy, clinical predictiveness, or pan-cancer universality. Residual proliferation, platform, lineage-composition, and other cell-line confounding remain relevant limitations.


In [ ]:
# =============================================================================
# Define notebook-603 stable artifact outputs
# =============================================================================

artifact_outputs = {
    "phase6.603.replication_hypothesis_manifest": {
        "path": replication_manifest_path,
        "shape": list(
            replication_hypothesis_manifest.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.603.cell_line_overlap_manifest": {
        "path": cell_line_overlap_path,
        "shape": list(
            cell_line_overlap_manifest.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.603.primary_external_associations": {
        "path": primary_external_path,
        "shape": list(
            primary_external_associations.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.603.nonoverlap_external_associations": {
        "path": nonoverlap_external_path,
        "shape": list(
            nonoverlap_external_associations.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.603.replication_summary": {
        "path": replication_summary_path,
        "shape": list(
            replication_summary_final.shape
        ),
        "artifact_role": "handoff",
    },
    "phase6.603.analysis_metadata": {
        "path": analysis_metadata_path,
        "shape": None,
        "artifact_role": "metadata",
    },
}

print(
    "Stable notebook-603 outputs:",
    len(artifact_outputs),
)

In [ ]:
# =============================================================================
# Build notebook-603 artifact identities
# =============================================================================

from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)

artifact_identity_rows = []

for artifact_id, artifact in artifact_outputs.items():
    artifact_path = artifact["path"]

    artifact_identity_rows.append(
        {
            "artifact_id": artifact_id,
            "path": project_relative_path(
                artifact_path
            ),
            "artifact_role": artifact[
                "artifact_role"
            ],
            "shape": artifact["shape"],
            "size_bytes": artifact_path.stat().st_size,
            "sha256": calculate_sha256(
                artifact_path
            ),
        }
    )

artifact_identities = pd.DataFrame(
    artifact_identity_rows
)

display(artifact_identities)

In [ ]:
# =============================================================================
# Define notebook-603 artifact provenance
# =============================================================================

NOTEBOOK_603_PATH = (
    "notebooks/phase6_pharmacogenomic_contexts/"
    "603_cross_screen_replication.ipynb"
)

artifact_inputs = {
    "phase6.603.replication_hypothesis_manifest": [
        "phase6.600.gdsc_program_drug_associations",
        "phase6.600.cross_resource_compound_catalog",
        "phase6.600.drug_eligibility",
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.ctrp_analysis_universe",
        "phase6.600.prism_analysis_universe",
    ],
    "phase6.603.cell_line_overlap_manifest": [
        "phase6.603.replication_hypothesis_manifest",
        "phase6.600.program_score_universe",
        "phase6.600.gdsc_analysis_universe",
        "phase6.600.ctrp_analysis_universe",
        "phase6.600.prism_analysis_universe",
    ],
    "phase6.603.primary_external_associations": [
        "phase6.603.replication_hypothesis_manifest",
        "phase6.600.program_score_universe",
        "phase6.600.ctrp_analysis_universe",
        "phase6.600.prism_analysis_universe",
    ],
    "phase6.603.nonoverlap_external_associations": [
        "phase6.603.primary_external_associations",
        "phase6.603.replication_hypothesis_manifest",
    ],
    "phase6.603.replication_summary": [
        "phase6.603.primary_external_associations",
        "phase6.603.nonoverlap_external_associations",
        "phase6.601.drug_level_results",
        "phase6.602.drug_program_attribution_summary",
    ],
    "phase6.603.analysis_metadata": [
        "phase6.603.replication_hypothesis_manifest",
        "phase6.603.cell_line_overlap_manifest",
        "phase6.603.primary_external_associations",
        "phase6.603.nonoverlap_external_associations",
        "phase6.603.replication_summary",
    ],
}

print(
    "Artifact provenance definitions:",
    len(artifact_inputs),
)

In [ ]:
# =============================================================================
# Build notebook-603 artifact registry entries
# =============================================================================

artifact_identity_index = (
    artifact_identities
    .set_index("artifact_id")
)

registry_entries = {}

for artifact_id in artifact_outputs:
    identity = artifact_identity_index.loc[
        artifact_id
    ]

    registry_entries[artifact_id] = {
        "path": identity["path"],
        "phase": 6,
        "status": "frozen",
        "artifact_role": identity["artifact_role"],
        "producer": {
            "type": "notebook",
            "path": NOTEBOOK_603_PATH,
        },
        "shape": (
            identity["shape"]
            if isinstance(identity["shape"], list)
            else None
        ),
        "size_bytes": int(identity["size_bytes"]),
        "sha256": identity["sha256"],
        "inputs": [
            {
                "type": "artifact",
                "artifact_id": input_artifact_id,
            }
            for input_artifact_id in artifact_inputs[
                artifact_id
            ]
        ],
    }

print(
    "Registry entries prepared:",
    len(registry_entries),
)

display(
    pd.DataFrame(
        [
            {
                "artifact_id": artifact_id,
                "artifact_role": entry["artifact_role"],
                "n_inputs": len(entry["inputs"]),
            }
            for artifact_id, entry in registry_entries.items()
        ]
    )
)

In [ ]:
# =============================================================================
# Register frozen notebook-603 artifacts
# =============================================================================

from pancancer_epigenetics.utils.artifact_registry import (
    validate_artifact_registry,
)

updated_registry = json.loads(
    json.dumps(artifact_registry)
)

updated_registry["artifacts"].update(
    registry_entries
)

validate_artifact_registry(
    updated_registry
)

Paths.artifact_registry.write_text(
    json.dumps(
        updated_registry,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "Registered notebook-603 artifacts:",
    sum(
        artifact_id.startswith("phase6.603.")
        for artifact_id in updated_registry["artifacts"]
    ),
)
print(
    "Artifact registry:",
    project_relative_path(
        Paths.artifact_registry
    ),
)

In [ ]:
# =============================================================================
# Final notebook-603 local QA
# =============================================================================

def overlap_sets_are_consistent():
    rows = cell_line_overlap_manifest.loc[
        cell_line_overlap_manifest[
            "primary_evaluable"
        ]
    ]

    for row in rows.itertuples(index=False):
        gdsc_models = set(
            json.loads(row.gdsc_model_ids)
        )
        external_models = set(
            json.loads(row.external_model_ids)
        )
        shared_models = set(
            json.loads(row.shared_model_ids)
        )
        nonoverlap_models = set(
            json.loads(
                row.external_nonoverlap_model_ids
            )
        )

        if shared_models & nonoverlap_models:
            return False
        if shared_models | nonoverlap_models != external_models:
            return False
        if shared_models != gdsc_models & external_models:
            return False
        if len(shared_models) != row.shared_models:
            return False
        if len(nonoverlap_models) != row.external_nonoverlap_models:
            return False

    return True


manifest_keys = [
    "gdsc_drug_id",
    "external_resource",
    "external_drug_id",
    "program",
]

expected_primary = (
    replication_hypothesis_manifest.loc[
        replication_hypothesis_manifest[
            "primary_evaluable"
        ],
        manifest_keys,
    ]
)

observed_primary = (
    primary_external_associations[
        manifest_keys
    ]
)

primary_membership_check = (
    expected_primary.merge(
        observed_primary,
        on=manifest_keys,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )["_merge"]
    .eq("both")
    .all()
)

registered_603 = {
    artifact_id: artifact
    for artifact_id, artifact in (
        load_artifact_registry()[
            "artifacts"
        ].items()
    )
    if artifact_id.startswith(
        "phase6.603."
    )
}

qa_results = {
    "six_artifacts_registered": (
        len(registered_603) == 6
    ),
    "summary_complete": (
        replication_summary_final[
            "final_screen_status"
        ].notna().all()
    ),
    "primary_family_sizes": (
        primary_external_associations.groupby(
            "external_resource"
        ).size().to_dict()
        == {
            "CTRP": 79,
            "PRISM": 98,
        }
    ),
    "nonoverlap_family_sizes": (
        nonoverlap_external_associations.groupby(
            "external_resource"
        ).size().to_dict()
        == {
            "CTRP": 64,
            "PRISM": 12,
        }
    ),
    "primary_membership_exact": (
        primary_membership_check
    ),
    "overlap_sets_consistent": (
        overlap_sets_are_consistent()
    ),
    "screen_replicated_count": (
        replication_summary_final[
            "final_screen_status"
        ].eq("SCREEN_REPLICATED").sum()
        == 37
    ),
    "nonoverlap_corroborated_count": (
        nonoverlap_external_associations[
            "nonoverlap_model_corroboration"
        ].sum()
        == 4
    ),
    "dual_screen_replication_count": (
        dual_screen_status[
            "dual_external_status"
        ].notna().sum()
        == 0
    ),
}

display(
    pd.Series(
        qa_results,
        name="PASS",
    )
)

assert all(qa_results.values())